# Table 3 单例：CG 与 Richardson 迭代数随 `top_q` 对比

固定 **Greengard Table 3** 中一例：`σ=0.1`，`N=3·10⁶`，Matérn 3/2（`Mat32`），`ε=10⁻⁵`，`L2_SCALED=True`，与 `benchmark_table3_table4_greengard.ipynb` 一致；**对多个 `top_q` 逐行出表**（默认 `0,16,32,48,64`，与主 notebook 的 `TOP_Q_SWEEP` 一致；试跑 `RUN_QUICK=True` 时只跑子集）。

对每个 `top_q`：用同一预计算状态分别跑 **CG** 与 **Richardson**。Richardson 在密 `η` 网格 + 多组 pilot 上筛选，并**额外并入偏小 `η`**（大 `η` 易数值发散）。终端 **print** 汇总表：`σ, ε, N, top_q, m, solver, iters, tot_s, relres, RMSE_y, RMSE_f`（`RMSE_y` 对测试点噪声标签，`RMSE_f` 对真值 `f(x)`）。不写文件。

In [9]:
import sys
import math
import time
import importlib
from pathlib import Path

import numpy as np

root = Path.cwd()
parents = [root, root.parent, root.parent.parent]
for base in parents:
    if (base / "efgp_eigenpro_py").exists():
        sys.path.insert(0, str(base))
        break
else:
    raise RuntimeError("cannot find efgp_eigenpro_py from current path")

import efgp_eigenpro_py.efgp_solver as efgp_solver
importlib.reload(efgp_solver)

from efgp_eigenpro_py.efgp_solver import EFGPSolver
from efgp_eigenpro_py.kernels import make_matern
from efgp_eigenpro_py.linear_solvers import richardson, tune_richardson_eta
from efgp_eigenpro_py import benchmark as bm
importlib.reload(bm)

<module 'efgp_eigenpro_py.benchmark' from 'd:\\NU\\ML\\efgp_eigenpro_py\\benchmark.py'>

In [10]:
# --- 与 Table 3 notebook 对齐 ---
DIM = 2
WAVEVEC = np.array([3.0, 4.0], dtype=float)
PHASE = 1.3
VARIANCE = 1.0
LENGTHSCALE = 0.1
NU_MATERN = 1.5
KERNEL = make_matern(lengthscale=LENGTHSCALE, dim=DIM, nu=NU_MATERN, variance=VARIANCE)

SIGMA = 0.1
EPS = 1e-5
L2_SCALED = True
TRAIN_CHUNK_SIZE = 2_000_000

RUN_QUICK = False
if RUN_QUICK:
    N_VALUE = 200_000
    TOP_Q_LIST = [0, 8, 16]
else:
    N_VALUE = 3_000_000
    TOP_Q_LIST = [0, 16, 32, 48, 64]

N_LIST_REF = [3_000_000, 10_000_000, 30_000_000]
BASE_SEED = 2026


def seed_for_n_table3(n_value: int) -> int:
    if n_value in N_LIST_REF:
        idx = N_LIST_REF.index(n_value)
    else:
        idx = int(round(math.log10(max(n_value, 1))))
    return int(BASE_SEED + idx)


SEED = seed_for_n_table3(N_VALUE)

CG_MAXITER_BASELINE = 100_000
CG_MAXITER_PRECOND = 50_000

EIG_METHOD = "eigsh"
EIG_TOL = 1e-1
EIG_MAXITER = 50
EIG_BLOCK_SIZE = None
EIG_OVERSAMPLE = 2

NTRGS_PER_D = 1000  # 与 benchmark_table3_table4_greengard 测试格一致

# Richardson：较密的 eta 网格 + 多组 pilot；另加偏小 eta（接近 1 的步长易炸）
ETA_GRID = np.unique(
    np.concatenate(
        [
            np.linspace(0.02, 0.30, 22),
            np.linspace(0.30, 0.70, 22),
            np.linspace(0.70, 1.10, 18),
        ]
    )
)
ETA_GRID_EXTRA = np.unique(
    np.concatenate([np.linspace(0.004, 0.12, 28), np.geomspace(0.003, 0.09, 18)])
)
ETA_PILOT_ITERS_LIST = [32, 64, 128, 192, 256]
TOPK_PILOT_ETAS = 4

print("N=", N_VALUE, "sigma=", SIGMA, "eps=", EPS, "seed=", SEED, "top_q=", TOP_Q_LIST)
print("eta_grid=", len(ETA_GRID), "+ extra=", len(ETA_GRID_EXTRA), " pilot_iters=", ETA_PILOT_ITERS_LIST)

N= 3000000 sigma= 0.1 eps= 1e-05 seed= 2026 top_q= [0, 16, 32, 48, 64]
eta_grid= 60 + extra= 46  pilot_iters= [32, 64, 128, 192, 256]


In [11]:
def f_eval(x: np.ndarray) -> np.ndarray:
    return np.cos(2.0 * math.pi * (x @ WAVEVEC) + PHASE)


def equispaced_grid(dim: int, n_per_d: int) -> np.ndarray:
    grid_1d = np.linspace(0.0, 1.0, n_per_d, endpoint=False)
    mesh = np.meshgrid(*([grid_1d] * dim), indexing="ij")
    return np.stack([g.reshape(-1) for g in mesh], axis=1)


def rms(a: np.ndarray, b: np.ndarray) -> float:
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    if not np.all(np.isfinite(d)):
        return float("nan")
    return float(np.sqrt(np.mean(d**2)))


X_TRG = equispaced_grid(DIM, NTRGS_PER_D)
Y_TRG_TRUE = f_eval(X_TRG)
_rng_trg = np.random.default_rng(SEED + 999)
Y_TRG_NOISY = Y_TRG_TRUE + SIGMA * _rng_trg.standard_normal(Y_TRG_TRUE.shape[0])


def generate_train_chunks(dim: int, n_value: int, chunk_size: int, seed: int, sigma: float):
    rng = np.random.default_rng(seed)
    produced = 0
    while produced < n_value:
        m = min(chunk_size, n_value - produced)
        x = rng.uniform(0.0, 1.0, size=(m, dim))
        f = f_eval(x)
        y = f + sigma * rng.standard_normal(m)
        yield x, y
        produced += m


def precompute_streaming(n_value: int, sigma: float, eps: float, seed: int):
    reg_lambda = sigma**2
    nufft_eps = eps / 10.0
    solver = EFGPSolver(
        KERNEL,
        reg_lambda=reg_lambda,
        eps=eps,
        nufft_tol=nufft_eps,
        l2scaled=L2_SCALED,
    )
    chunk_iter = generate_train_chunks(DIM, n_value, TRAIN_CHUNK_SIZE, seed, sigma)
    bounds = (np.zeros(DIM), np.ones(DIM))
    t0 = time.perf_counter()
    state = solver.precompute_streaming(chunk_iter, n_total=int(n_value), x_bounds=bounds)
    t1 = time.perf_counter()
    return solver, state, t1 - t0


print("precomputing (one pass for this eps)...")
t_pre0 = time.perf_counter()
SOLVER, STATE, PRE_TIME = precompute_streaming(N_VALUE, SIGMA, EPS, SEED)
t_pre1 = time.perf_counter()
m_grid = (STATE.grid.mtot - 1) // 2
print(f"precompute_wall_s={t_pre1 - t_pre0:.3f} (reported streaming {PRE_TIME:.3f})  m={m_grid}  M={STATE.rhs.size}")

precomputing (one pass for this eps)...
precompute_wall_s=0.565 (reported streaming 0.565)  m=94  M=35721


In [12]:
def _merge_richardson_eta_candidates(matvec, b, x0, precond, maxiter: int):
    seen = set()
    order = []
    for pcap in ETA_PILOT_ITERS_LIST:
        cap = min(int(pcap), maxiter)
        best_eta, records = tune_richardson_eta(
            matvec,
            b,
            x0,
            ETA_GRID,
            EPS,
            maxiter,
            precond=precond,
            pilot_iters=cap,
            relres_check_every=1,
        )
        rec_sorted = sorted(records, key=lambda t: (t[2], t[1]))
        picked = [best_eta] + [t[0] for t in rec_sorted[:TOPK_PILOT_ETAS]]
        for e in picked:
            e = float(e)
            if e not in seen:
                seen.add(e)
                order.append(e)
    for e in ETA_GRID_EXTRA:
        e = float(e)
        if e not in seen:
            seen.add(e)
            order.append(e)
    return order


def _pick_best_richardson_full(matvec, b, x0, etas, precond, maxiter: int):
    """完整 Richardson：先收敛优先，再有限 relres，再对有限 beta 用 matvec 重算残差；避免在全员 relres=inf 时误选“最少步数”的爆炸解。"""
    if not etas:
        raise ValueError("Richardson: 无 eta 候选")
    tol_ok = EPS * (1.0 + 1e-9)
    norm_b = max(float(np.linalg.norm(b)), 1e-300)
    etas_try = sorted({float(e) for e in etas})
    candidates = []
    for eta in etas_try:
        beta, it, rr = richardson(
            matvec,
            b,
            x0,
            float(eta),
            EPS,
            maxiter,
            precond=precond,
            relres_check_every=1,
        )
        candidates.append((int(it), float(rr), float(eta), np.array(beta, copy=True)))
    conv = [c for c in candidates if np.isfinite(c[1]) and c[1] <= tol_ok]
    if conv:
        conv.sort(key=lambda c: (c[0], c[1]))
        it, rr, eta, beta = conv[0]
        return it, rr, eta, len(etas_try), beta
    finite_rr = [c for c in candidates if np.isfinite(c[1])]
    if finite_rr:
        finite_rr.sort(key=lambda c: (c[1], c[0]))
        it, rr, eta, beta = finite_rr[0]
        return it, rr, eta, len(etas_try), beta
    measured = []
    for it, _rr, eta, beta in candidates:
        if not np.all(np.isfinite(beta)):
            continue
        resid = matvec(beta) - b
        if not np.all(np.isfinite(resid)):
            continue
        rr_m = float(np.linalg.norm(resid) / norm_b)
        if np.isfinite(rr_m):
            measured.append((it, rr_m, eta, beta))
    if measured:
        measured.sort(key=lambda c: (c[1], c[0]))
        it, rr_m, eta, beta = measured[0]
        return it, rr_m, eta, len(etas_try), beta
    beta0 = np.zeros_like(b, dtype=np.result_type(b.dtype, np.complex128))
    return int(maxiter), float("inf"), float("nan"), len(etas_try), beta0


def _fmt_rel(rr: float) -> str:
    if not np.isfinite(rr):
        return "   inf  "
    return f"{rr:8.2e}"


def _print_summary_table(rows):
    hdr = (
        f"{'solver':>10} {'sigma':>9} {'eps':>9} {'N':>10} {'top_q':>5} {'m':>4} {'iters':>7} {'tot_s':>9} {'relres':>10} {'RMSE_y':>10} {'RMSE_f':>10} {'eta':>8}"
    )
    print(hdr)
    print("-" * len(hdr))
    for r in rows:
        eta_s = f"{r['eta']:8.4f}" if r.get("eta") is not None else "     nan"
        print(
            f"{r['solver']:>10} {r['sigma']:9.2e} {r['eps']:9.1e} {r['N']:10d} {r['top_q']:5d} {r['m']:4d} {r['iters']:7d} {r['tot_s']:9.2f} {_fmt_rel(r['relres']):>10} {r['RMSE_y']:10.4e} {r['RMSE_f']:10.4e} {eta_s}"
        )


matvec = lambda v: SOLVER._apply_A(STATE, v)
b = STATE.rhs
x0 = np.zeros_like(b)
summary = []

for top_q in TOP_Q_LIST:
    precond_obj, _eigpairs, _mu, t_eig, t_pc, top_q_used = bm.build_precond_from_state(
        SOLVER,
        STATE,
        top_q,
        eig_method=EIG_METHOD,
        eig_tol=EIG_TOL,
        eig_maxiter=EIG_MAXITER,
        eig_block_size=EIG_BLOCK_SIZE,
        eig_oversample=EIG_OVERSAMPLE,
    )
    precond_apply = precond_obj.apply if precond_obj is not None else None
    maxiter = CG_MAXITER_BASELINE if top_q_used == 0 else CG_MAXITER_PRECOND

    t0 = time.perf_counter()
    beta_cg, it_cg, rel_cg, _h, _stats = bm.pcg_solve(
        matvec,
        b,
        tol=EPS,
        maxiter=maxiter,
        precond=precond_apply,
        store_history=False,
        return_stats=True,
    )
    t_cg = time.perf_counter() - t0
    tp0 = time.perf_counter()
    yhat_cg = SOLVER.predict(X_TRG, beta_cg, STATE)
    tp1 = time.perf_counter()
    t_pred_cg = tp1 - tp0
    tot_cg = PRE_TIME + float(t_eig) + t_cg + t_pred_cg

    t_tune0 = time.perf_counter()
    eta_list = _merge_richardson_eta_candidates(matvec, b, x0, precond_apply, maxiter)
    t_tune1 = time.perf_counter()
    t_tune = t_tune1 - t_tune0

    t1 = time.perf_counter()
    it_r, rel_r, eta_win, n_eta, beta_r = _pick_best_richardson_full(
        matvec, b, x0, eta_list, precond_apply, maxiter
    )
    t_r = time.perf_counter() - t1
    tp0 = time.perf_counter()
    yhat_r = SOLVER.predict(X_TRG, beta_r, STATE)
    tp1 = time.perf_counter()
    t_pred_r = tp1 - tp0
    tot_r = PRE_TIME + float(t_eig) + t_tune + t_r + t_pred_r

    summary.append(
        {
            "solver": "CG",
            "sigma": float(SIGMA),
            "eps": float(EPS),
            "N": int(N_VALUE),
            "top_q": int(top_q_used),
            "m": int(m_grid),
            "iters": int(it_cg),
            "tot_s": float(tot_cg),
            "relres": float(rel_cg),
            "RMSE_y": rms(Y_TRG_NOISY, yhat_cg),
            "RMSE_f": rms(Y_TRG_TRUE, yhat_cg),
            "eta": None,
        }
    )
    summary.append(
        {
            "solver": "Richardson",
            "sigma": float(SIGMA),
            "eps": float(EPS),
            "N": int(N_VALUE),
            "top_q": int(top_q_used),
            "m": int(m_grid),
            "iters": int(it_r),
            "tot_s": float(tot_r),
            "relres": float(rel_r),
            "RMSE_y": rms(Y_TRG_NOISY, yhat_r),
            "RMSE_f": rms(Y_TRG_TRUE, yhat_r),
            "eta": float(eta_win),
        }
    )
    print(
        f"[done] top_q={top_q_used}  CG it={it_cg} rel={rel_cg:.2e} | R it={it_r} rel={rel_r} eta={eta_win:.4g} (etas tried full={n_eta}, tune {t_tune:.1f}s)"
    )

print()
_print_summary_table(summary)
print()
print(
    "Richardson: main eta grid", len(ETA_GRID), "+ extra", len(ETA_GRID_EXTRA), "pilots", ETA_PILOT_ITERS_LIST
)
print("tot_s = precompute + eig + (tune+)solve + predict ；RMSE_y 对噪声测试标签，RMSE_f 对真值 f(x)")

C:\Users\24681\AppData\Local\Temp\ipykernel_27992\4177084436.py:12: RuntimeWarning: overflow encountered in square
  return float(np.sqrt(np.mean((a - b) ** 2)))


[done] top_q=0  CG it=2851 rel=7.79e-06 | R it=58 rel=inf eta=0.9824 (etas tried full=54, tune 135.9s)


C:\Users\24681\AppData\Local\Temp\ipykernel_27992\4177084436.py:12: RuntimeWarning: overflow encountered in square
  return float(np.sqrt(np.mean((a - b) ** 2)))


[done] top_q=16  CG it=1940 rel=9.47e-06 | R it=64 rel=inf eta=0.9588 (etas tried full=54, tune 173.1s)


c:\Users\24681\anaconda3\Lib\site-packages\numpy\linalg\linalg.py:2550: RuntimeWarning: overflow encountered in scalar add
  sqnorm = x_real.dot(x_real) + x_imag.dot(x_imag)
C:\Users\24681\AppData\Local\Temp\ipykernel_27992\4177084436.py:12: RuntimeWarning: overflow encountered in square
  return float(np.sqrt(np.mean((a - b) ** 2)))


[done] top_q=32  CG it=1291 rel=9.04e-06 | R it=69 rel=inf eta=1.006 (etas tried full=54, tune 208.5s)


c:\Users\24681\anaconda3\Lib\site-packages\numpy\linalg\linalg.py:2550: RuntimeWarning: overflow encountered in scalar add
  sqnorm = x_real.dot(x_real) + x_imag.dot(x_imag)
C:\Users\24681\AppData\Local\Temp\ipykernel_27992\4177084436.py:12: RuntimeWarning: overflow encountered in square
  return float(np.sqrt(np.mean((a - b) ** 2)))


[done] top_q=48  CG it=986 rel=8.77e-06 | R it=73 rel=inf eta=1.006 (etas tried full=54, tune 237.6s)
[done] top_q=64  CG it=818 rel=8.52e-06 | R it=77 rel=inf eta=1.029 (etas tried full=54, tune 271.8s)

    solver     sigma       eps          N top_q    m   iters     tot_s     relres     RMSE_y     RMSE_f      eta
----------------------------------------------------------------------------------------------------------------
        CG  1.00e-01   1.0e-05    3000000     0   94    2851     20.76   7.79e-06 1.0025e-01 5.4777e-03      nan
Richardson  1.00e-01   1.0e-05    3000000     0   94      58    173.17      inf          inf        inf   0.9824
        CG  1.00e-01   1.0e-05    3000000    16   94    1940     17.45   9.47e-06 1.0025e-01 5.4782e-03      nan
Richardson  1.00e-01   1.0e-05    3000000    16   94      64    221.61      inf          inf        inf   0.9588
        CG  1.00e-01   1.0e-05    3000000    32   94    1291     15.36   9.04e-06 1.0025e-01 5.4781e-03      nan
Rich

C:\Users\24681\AppData\Local\Temp\ipykernel_27992\4177084436.py:12: RuntimeWarning: overflow encountered in square
  return float(np.sqrt(np.mean((a - b) ** 2)))


**说明**：`tot_s` 含预计算、`eig`（`top_q>0`）、Richardson 的 `tune`、求解与 `predict`。**`173.17` 这类大数是 Richardson 行的总墙钟时间**（`tune` 很慢），不是 relres。若曾出现 **`relres` 与 RMSE 全是 `inf`**：旧逻辑在「所有试跑的迭代内 `relres` 都变成 `inf`」时，会误选**步数最少**的那次（往往是最早数值爆炸的解），`beta` 里含 NaN/极大值，`predict` 后 RMSE 平方会溢出。现已改为：**优先收敛 → 优先有限 relres → 否则对有限 `beta` 用 `||b-Aβ||/||b||` 重算**；仍失败则 `beta=0`、RMSE 为 `nan`。另：**带 EigenPro 预条件时，Richardson 不像 CG 那样保证下降**，可能对所有 `η` 都不稳定，属算法限制而非纯实现 bug。